In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from feature_engine.scaling import MeanNormalizationScaler
from feature_engine.encoding import OneHotEncoder
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
import joblib

In [2]:
df = pd.read_csv("../data/hour_clean.csv")

df = df.drop(columns=["instant", "dteday", "casual", "registered"])

input_features = df.drop(columns=["cnt"])
bike_demand = df["cnt"]

print("Input Features: ")
display(input_features.head())

Input Features: 


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0


In [7]:
train_features, test_features, train_target, test_target = train_test_split(
    input_features,
    bike_demand,
    test_size=0.2,
    shuffle=False
)

numeric_features = ["temp", "atemp", "hum", "windspeed"]
categorical_features = [
    "season", "yr", "mnth", "hr", "weekday",
    "workingday", "weathersit", "holiday"
]

# Scale numeric features
numeric_scaler = MeanNormalizationScaler(variables=numeric_features)
train_features = numeric_scaler.fit_transform(train_features)
test_features = numeric_scaler.transform(test_features)

# Ensure categorical features are categorical dtype
train_features[categorical_features] = train_features[categorical_features].astype("category")
test_features[categorical_features] = test_features[categorical_features].astype("category")

# One-hot encode categoricals
encoder = OneHotEncoder(
    variables=categorical_features,
    drop_last=True
)

train_features = encoder.fit_transform(train_features)
test_features = encoder.transform(test_features)

In [ ]:
# Model LGBM
lgbm_model = LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42,
    force_row_wise=True
)

lgbm_model.fit(train_features, train_target)

# Model Simple Linear Regression
single_feature = train_features[["temp"]]
linear_model = LinearRegression()
linear_model.fit(single_feature, train_target)

# Model Multiple Linear Regression
linear_features = [
    "temp",
    "atemp",
    "hum",
    "windspeed"
]

feature_train_linear = train_features[linear_features]
feature_test_linear = test_features[linear_features]

multiple_linear_model = LinearRegression()
multiple_linear_model.fit(feature_train_linear, train_target)

# Model KNN Regression
knn_features = [
    "temp",
    "atemp",
    "hum",
    "windspeed"
]

feature_train_knn = train_features[knn_features].copy()
feature_test_knn = test_features[knn_features].copy()

knn_scaler = MeanNormalizationScaler(variables=knn_features)
feature_train_knn = knn_scaler.fit_transform(feature_train_knn)
feature_test_knn = knn_scaler.transform(feature_test_knn)

knn_model = KNeighborsRegressor(
    n_neighbors=10,
    weights="distance"
)

knn_model.fit(feature_train_knn, train_target)

import joblib

artifacts = {
    # Models
    "lgbm": lgbm_model,
    "simple_linear": linear_model,
    "multiple_linear": multiple_linear_model,
    "knn": knn_model,

    # Preprocessing
    "numeric_scaler": numeric_scaler,     
    "encoder": encoder,                   
    "knn_scaler": knn_scaler,    
    
    #Feature list
    "features": {
        "simple_linear": ["temp"],
        "multiple_linear": ["temp", "atemp", "hum", "windspeed"],
        "knn": ["temp", "atemp", "hum", "windspeed"],
        "lgbm_columns": train_features.columns.tolist()
    },
    "target": "cnt",
    "model_version": "1.0",          
}

joblib.dump(artifacts, "models_bundle.pkl")


[LightGBM] [Info] Total Bins 324
[LightGBM] [Info] Number of data points in the train set: 13903, number of used features: 53
[LightGBM] [Info] Start training from score 174.639143


['models_bundle.pkl']

In [10]:
def evaluate_regression(y_true, y_pred, name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    print(f"{name}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R²: {r2:.3f}")
    print("-" * 30)

    return rmse, r2

In [11]:
results = {}

results["LightGBM"] = evaluate_regression(
    test_target,
    lgbm_model.predict(test_features),
    "LightGBM"
)

results["Simple Linear"] = evaluate_regression(
    test_target,
    linear_model.predict(test_features[["temp"]]),
    "Simple Linear"
)

results["Multiple Linear"] = evaluate_regression(
    test_target,
    multiple_linear_model.predict(feature_test_linear),
    "Multiple Linear"
)

results["KNN"] = evaluate_regression(
    test_target,
    knn_model.predict(feature_test_knn),
    "KNN"
)

metrics = {
    model: {"RMSE": rmse, "R2": r2}
    for model, (rmse, r2) in results.items()
}

artifacts["metrics"] = metrics
joblib.dump(artifacts, "models_bundle.pkl")


LightGBM
  RMSE: 75.98
  R²: 0.881
------------------------------
Simple Linear
  RMSE: 219.16
  R²: 0.012
------------------------------
Multiple Linear
  RMSE: 209.54
  R²: 0.097
------------------------------
KNN
  RMSE: 225.92
  R²: -0.050
------------------------------


['models_bundle.pkl']